# 05. Итоговый отчёт — D4v2 FAQ Architecture Experiment

**Исследовательский вопрос**: какой способ предоставления знаний LLM
минимально достаточен для FAQ-модуля клиники?

**Содержание**:
1. Дизайн эксперимента
2. Сводные метрики по стратегиям
3. Статистический анализ
4. Retrieval-метрики (S2-S4)
5. Анализ ошибок
6. Decision memo
7. Backend follow-up

**Артефакты**: `outputs/reports/d4_final_report.md`, `outputs/error_table.csv`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../..')

import os
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from d4.config import load_config

sns.set_style('whitegrid')

OUTPUTS = Path('../outputs')
REPORTS = OUTPUTS / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

config = load_config('../configs/experiment.yaml')

## 1. Дизайн эксперимента

In [ ]:
print('='*60)
print('D4v2: Unified Pipeline FAQ Architecture Experiment')
print('='*60)
print(f'\nМодель LLM: {config.llm.model}')
print(f'Провайдер LLM: {config.llm.provider}')
print(f'Модель Judge: {config.judge.model}')
print(f'Embedding: {config.embedding.model}')
print(f'Temperature: {config.llm.temperature}')
print(f'Top-k: {config.embedding.top_k}')
print(f'Seed: {config.experiment.seed}')
print()
print('Стратегии:')
for s in config.strategies:
    print(f'  {s["id"]}: {s["name"]}')
print()
print('Pre-registered endpoints:')
print(f'  Primary: {config.statistics["primary_endpoint"]}')
print(f'  Secondary: {config.statistics["secondary_endpoint"]}')

D4v2: Unified Pipeline FAQ Architecture Experiment

Модель LLM: qwen/qwen3.5-35b-a3b


KeyError: 'judge_llm'

## 2. Сводные метрики

In [ ]:
# Загрузка summary из 03_evaluation
df_summary = pd.read_csv(OUTPUTS / 'metrics_summary.csv', index_col=0)

# Ключевые столбцы для отчёта
key_cols = [
    'answerability_accuracy',
    'doctor_match_rate',
    'specialization_match_rate',
    'unsupported_claim_rate',
    'judge_mean_factual_accuracy',
    'judge_hallucination_rate',
    'sys_latency_p50',
    'sys_latency_p95',
    'sys_mean_context_length',
]

available_cols = [c for c in key_cols if c in df_summary.columns]
print('Сводная таблица метрик:')
df_summary[available_cols].round(3)

In [ ]:
# Radar chart: сравнение стратегий
radar_metrics = ['answerability_accuracy', 'doctor_match_rate', 'specialization_match_rate']
radar_metrics = [m for m in radar_metrics if m in df_summary.columns]

if len(radar_metrics) >= 3:
    from math import pi
    
    categories = radar_metrics
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    strategy_colors = {'S1': '#2196F3', 'S2': '#4CAF50', 'S3': '#FF9800', 'S4': '#9C27B0', 'B0': '#757575'}
    
    for sid in df_summary.index:
        values = df_summary.loc[sid, categories].values.tolist()
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, 
                color=strategy_colors.get(sid, 'gray'), label=sid)
        ax.fill(angles, values, alpha=0.1, color=strategy_colors.get(sid, 'gray'))
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([m.replace('_', '\n') for m in categories], fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.set_title('Сравнение стратегий (deterministic метрики)', pad=20)
    
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'figures' / 'radar_strategies.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Недостаточно метрик для radar chart')

## 3. Статистический анализ

In [ ]:
# Загрузка парных тестов
pairwise_path = OUTPUTS / 'pairwise_tests.csv'
if pairwise_path.exists():
    df_paired = pd.read_csv(pairwise_path)
    print('Парные сравнения (Wilcoxon + Holm):')
    print(df_paired.to_string(index=False))
    
    # Интерпретация
    print('\nИнтерпретация:')
    sig_pairs = df_paired[df_paired['sig'] == '✓']
    if len(sig_pairs) == 0:
        print('  Нет статистически значимых различий между стратегиями.')
        print('  → Practical decision rule: выбираем САМЫЙ ПРОСТОЙ сценарий (S1).')
    else:
        print(f'  Значимые различия: {len(sig_pairs)} пар')
        for _, row in sig_pairs.iterrows():
            better = row['A'] if row['diff'] > 0 else row['B']
            print(f'    {row["A"]} vs {row["B"]}: {better} лучше (d={row["d"]}, p_holm={row["p_holm"]})')
else:
    print('pairwise_tests.csv не найден — запустите 03_evaluation')

## 4. Retrieval-метрики (S2-S4)

In [ ]:
# Retrieval метрики из summary
ret_cols = [c for c in df_summary.columns if c.startswith('ret_')]

if ret_cols:
    print('Retrieval-метрики (S2-S4):')
    retrieval_strategies = [s for s in df_summary.index if s in ('S2', 'S3', 'S4')]
    print(df_summary.loc[retrieval_strategies, ret_cols].round(3))
    
    # Визуализация
    fig, ax = plt.subplots(figsize=(10, 5))
    df_ret_plot = df_summary.loc[retrieval_strategies, ret_cols].T
    df_ret_plot.plot.bar(ax=ax, edgecolor='black', alpha=0.8)
    ax.set_title('Retrieval-метрики по стратегиям')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Стратегия')
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'figures' / 'retrieval_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Retrieval-метрики отсутствуют (gold_map не заполнен)')

## 5. Анализ ошибок

In [ ]:
from d4.pipeline.orchestrator import Orchestrator
from d4.data_gen.query_generator import load_eval_set

results = Orchestrator.load_results(OUTPUTS / 'raw_results.jsonl')
eval_path = Path('../data/eval_set.yaml')
if not eval_path.exists():
    eval_path = Path('../data/eval_set_raw.yaml')
samples = load_eval_set(eval_path)
sample_map = {s.sample_id: s for s in samples}

# Собираем ошибки: неправильная answerability или unsupported claims
errors = []
for r in results:
    sample = sample_map.get(r.sample_id)
    if not sample:
        continue
    
    is_error = False
    error_types = []
    
    # Answerability mismatch
    if r.answer.answerable != sample.answerable:
        is_error = True
        error_types.append('answerability_mismatch')
    
    # API error
    if r.error:
        is_error = True
        error_types.append('api_error')
    
    if is_error:
        errors.append({
            'sample_id': r.sample_id,
            'strategy': r.strategy_id.value,
            'category': sample.category,
            'query': sample.query[:80],
            'error_types': ', '.join(error_types),
            'expected_answerable': sample.answerable,
            'got_answerable': r.answer.answerable,
            'answer_snippet': r.answer.answer[:100],
        })

df_errors = pd.DataFrame(errors)
print(f'Всего ошибок: {len(df_errors)} из {len(results)} результатов')

if len(df_errors) > 0:
    print(f'\nОшибки по стратегиям:')
    print(df_errors['strategy'].value_counts())
    print(f'\nОшибки по категориям:')
    print(df_errors['category'].value_counts())
    print(f'\nОшибки по типам:')
    print(df_errors['error_types'].value_counts())

In [ ]:
# Сохранение error table
if len(df_errors) > 0:
    df_errors.to_csv(OUTPUTS / 'error_table.csv', index=False)
    print(f'Сохранено: {OUTPUTS / "error_table.csv"}')
    print('\nТоп-10 ошибок:')
    df_errors.head(10)[['sample_id', 'strategy', 'category', 'error_types', 'query']]
else:
    print('Ошибок нет!')

In [ ]:
# Heatmap ошибок: category × strategy
if len(df_errors) > 0:
    error_pivot = df_errors.pivot_table(
        values='sample_id', 
        index='category', 
        columns='strategy', 
        aggfunc='count',
        fill_value=0,
    )
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(error_pivot, annot=True, fmt='d', cmap='Reds', ax=ax)
    ax.set_title('Количество ошибок: категория × стратегия')
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'figures' / 'error_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Decision Memo

In [ ]:
# Автоматическая генерация decision memo
print('='*60)
print('DECISION MEMO — D4v2 FAQ Architecture Experiment')
print('='*60)

# Определяем лучшую стратегию по primary endpoint
primary = 'answerability_accuracy'
if primary in df_summary.columns:
    # S1-S4 (без B0)
    llm_strategies = [s for s in df_summary.index if s != 'B0']
    best = df_summary.loc[llm_strategies, primary].idxmax()
    best_val = df_summary.loc[best, primary]
    
    s1_val = df_summary.loc['S1', primary] if 'S1' in df_summary.index else 0
    
    print(f'\n1. Primary endpoint ({primary}):')
    for sid in llm_strategies:
        val = df_summary.loc[sid, primary]
        marker = ' ← best' if sid == best else ''
        print(f'   {sid}: {val:.3f}{marker}')
    
    print(f'\n2. Practical decision rule:')
    print(f'   "Выбрать самый простой сценарий, не уступающий materially"')
    
    # Проверяем: есть ли значимые различия с S1?
    if pairwise_path.exists():
        df_p = pd.read_csv(pairwise_path)
        s1_sig = df_p[(df_p['A'] == 'S1') & (df_p['sig'] == '✓')]
        
        if len(s1_sig) == 0:
            print(f'\n   → S1 (Full Context) НЕ уступает другим стратегиям значимо.')
            print(f'   → РЕКОМЕНДАЦИЯ: S1 (Full Context) — самый простой,')
            print(f'     не требует retrieval layer, минимальная сложность.')
            recommendation = 'S1'
        else:
            better_than_s1 = s1_sig[s1_sig['diff'] < 0]['B'].tolist()
            if better_than_s1:
                rec = better_than_s1[0]  # Самая простая из лучших
                print(f'\n   → Стратегии {better_than_s1} значимо лучше S1.')
                print(f'   → РЕКОМЕНДАЦИЯ: {rec}')
                recommendation = rec
            else:
                print(f'\n   → S1 значимо лучше некоторых, не хуже остальных.')
                print(f'   → РЕКОМЕНДАЦИЯ: S1 (Full Context)')
                recommendation = 'S1'
    else:
        recommendation = best
        print(f'\n   → Статистические тесты не проведены. Best by value: {best}')
    
    print(f'\n3. B0 (baseline без LLM):')
    if 'B0' in df_summary.index:
        b0_val = df_summary.loc['B0', primary]
        print(f'   B0 accuracy: {b0_val:.3f} vs {recommendation}: {df_summary.loc[recommendation, primary]:.3f}')
        print(f'   LLM advantage: +{df_summary.loc[recommendation, primary] - b0_val:.3f}')
    
    print(f'\n4. ИТОГОВАЯ РЕКОМЕНДАЦИЯ: {recommendation}')
else:
    print('Primary endpoint не найден в summary')
    recommendation = 'N/A'

## 7. Backend Follow-up

In [ ]:
print('BACKEND FOLLOW-UP')
print('='*60)
print()
print(f'Рекомендованная стратегия: {recommendation}')
print()

if recommendation == 'S1':
    print('Архитектурные следствия:')
    print('  - pgvector: НЕ НУЖЕН (нет vector retrieval)')
    print('  - Postgres FTS: НЕ НУЖЕН (нет lexical retrieval)')
    print('  - Storage: KB в YAML файлах, загрузка при старте')
    print('  - Prompt: context stuffing, все данные в system prompt')
    print('  - Сложность: минимальная, без дополнительных зависимостей')
elif recommendation in ('S2',):
    print('Архитектурные следствия:')
    print('  - pgvector: НЕ НУЖЕН')
    print('  - Postgres FTS: РЕКОМЕНДУЕТСЯ (BM25/tsvector)')
    print('  - Storage: KB в БД с индексами для полнотекстового поиска')
elif recommendation in ('S3',):
    print('Архитектурные следствия:')
    print('  - pgvector: НУЖЕН')
    print('  - Postgres FTS: не нужен')
    print('  - Embedding model: деплой bge-m3 или API')
elif recommendation in ('S4',):
    print('Архитектурные следствия:')
    print('  - pgvector: НУЖЕН')
    print('  - Postgres FTS: НУЖЕН')
    print('  - Hybrid retrieval: RRF fusion layer')
    print('  - Наибольшая сложность инфраструктуры')

## 8. Генерация финального отчёта (Markdown)

In [ ]:
report_lines = [
    '# D4v2: Итоговый отчёт — FAQ Architecture Experiment',
    '',
    '## 1. Дизайн',
    '',
    f'- **Модель LLM**: {config.llm.model}',
    f'- **Провайдер LLM**: {config.llm.provider}',
    f'- **Judge**: {config.judge.model}',
    f'- **Embedding**: {config.embedding.model}',
    f'- **Eval set**: {len(samples)} запросов',
    f'- **KB**: {sum(c.token_count for c in load_chunks("../data/kb/chunks.json"))} токенов',
    '',
    '### Стратегии',
    '',
    '| ID | Название | LLM |',
    '|----|----------|:---:|',
]

for s in config.strategies:
    llm_mark = '✓' if s.get('uses_llm', True) else '✗'
    report_lines.append(f'| {s["id"]} | {s["name"]} | {llm_mark} |')

report_lines.extend([
    '',
    '## 2. Сводные метрики',
    '',
    df_summary[available_cols].round(3).to_markdown(),
    '',
    '## 3. Статистический анализ',
    '',
])

if pairwise_path.exists():
    df_p = pd.read_csv(pairwise_path)
    report_lines.append(df_p.to_markdown(index=False))
else:
    report_lines.append('*Парные тесты не проведены*')

report_lines.extend([
    '',
    '## 4. Decision Memo',
    '',
    f'**Рекомендованная стратегия: {recommendation}**',
    '',
    'Practical decision rule: выбрать самый простой сценарий,',
    'не уступающий materially по primary endpoint.',
    '',
    '## 5. Backend Follow-up',
    '',
])

if recommendation == 'S1':
    report_lines.extend([
        '- pgvector: **не нужен**',
        '- Postgres FTS: **не нужен**',
        '- Архитектура: context stuffing, KB загружается при старте',
    ])
else:
    report_lines.append(f'- Требуется {recommendation}-специфичная инфраструктура')

report_text = '\n'.join(report_lines)

report_path = REPORTS / 'd4_final_report.md'
report_path.write_text(report_text, encoding='utf-8')
print(f'Отчёт сохранён: {report_path}')
print(f'Размер: {len(report_text)} символов')

In [ ]:
# Итоговая сводка артефактов
print('\n' + '='*60)
print('АРТЕФАКТЫ ЭКСПЕРИМЕНТА')
print('='*60)

artifacts = [
    ('outputs/raw_results.jsonl', 'Сырые результаты прогона'),
    ('outputs/judge_scores.json', 'Оценки LLM-judge'),
    ('outputs/metrics_summary.csv', 'Сводная таблица метрик'),
    ('outputs/pairwise_tests.csv', 'Парные статистические тесты'),
    ('outputs/error_table.csv', 'Таблица ошибок'),
    ('outputs/reports/d4_final_report.md', 'Финальный отчёт'),
]

for path, desc in artifacts:
    full = OUTPUTS.parent / path
    status = '✓' if full.exists() else '✗'
    print(f'  {status} {path} — {desc}')

print(f'\nФигуры:')
for fig_path in sorted((OUTPUTS / 'figures').glob('*.png')):
    print(f'  ✓ figures/{fig_path.name}')